# <center> Running a fixed-sample MLMC estimator</center> 



This notebook uses the synthetic model introduced in the [synthetic_linear_model](synthetic_linear_model.ipynb) notebook to introduce the `MLMCRunner`. The model, correction evaluator, statistics classes, and runner have deliberately separate responsibilities. Here we connect those pieces and run a complete multilevel estimator for the first time.

By the end, we will have:

1. constructed a user model satisfying `MultilevelModel`;
2. configured a linear-system solver and reproducible base seed;
3. selected fixed sample counts for every correction level;
4. run the MLMC estimator and interpreted its result;
5. repeated the run reproducibly; and
6. added new samples without discarding or repeating old samples.


## 1. MLMC Runner Workflow

The user supplies three configured objects or values: a multilevel model, a system solver, and a base seed. A fixed execution request additionally supplies the number of correction samples to calculate at each active level.

<pre>
MultilevelModel       SystemSolver       base_seed
       │                    │                 │
       └────────────────────┼─────────────────┘
                            │
                  samples_per_level
                            │
                            ▼
                       MLMCRunner
                            │
            for every (level, sample_index)
                            │
                            ▼
              compute_sample_correction()
                            │
                            ▼
                    LevelStatistics
                            │
                            ▼
                       MLMCResult
</pre>

The runner schedules and identifies samples. It does not decide how the random model is constructed, assemble a physical matrix itself, implement a linear solver, or retain solution vectors. For each correction it delegates the complete fine/coarse calculation to `compute_sample_correction()`, consumes the returned scalar correction and elapsed time, and then releases the solve results.


## 5. Choose fixed sample counts

For the first run, the user chooses the counts directly:

$$
(N_0,N_1,N_2,N_3)=(5 \times 10^4,\, 4000,\, 1000,\,200).
$$

The entry at position $\ell$ is the number of independent samples of $Y_\ell$. Thus, this request schedules 50,000 evaluations of $Y_0=Q_0$, but only 200 evaluations of $Y_3=Q_3-Q_2$.

The initial runner requires at least two samples at every selected level so each unbiased sample variance is defined. These counts are for MLMC samples, not iterations taken by the linear solver.

Because we will leave `finest_level=None`, the runner uses the model's finest available level:

$$
L=\texttt{number\_of\_levels}-1=3.
$$

Consequently, the count sequence must have exactly four entries and the returned estimator targets $\mathbb{E}[Q_3]$.


## 2. What quantity does the runner estimate?

Suppose the model provides approximations $Q_0,Q_1,\ldots,Q_L$, where level 0 is the coarsest level and level $L$ is the selected finest level. Define the correction variables

$$
Y_0=Q_0
$$

and

$$
Y_\ell=Q_\ell-Q_{\ell-1}, \qquad \ell>0.
$$

Adding all corrections from level 0 through level $L$ gives the telescoping identity

$$
Q_L
=Q_0+(Q_1-Q_0)+\cdots+(Q_L-Q_{L-1})
=\sum_{\ell=0}^{L}Y_\ell.
$$

Taking expectations gives

$$
\mathbb{E}[Q_L]
=\sum_{\ell=0}^{L}\mathbb{E}[Y_\ell].
$$

The fixed-sample MLMC estimator replaces each correction expectation with an independent sample mean:

$$
\widehat Q_{\mathrm{ML}}
=\sum_{\ell=0}^{L}\overline{Y}_\ell,
\qquad
\overline{Y}_\ell
=\frac{1}{N_\ell}\sum_{i=0}^{N_\ell-1}Y_\ell^{(i)}.
$$

The sample counts $N_\ell$ may differ by level. MLMC normally uses many inexpensive coarse corrections and fewer expensive fine corrections. Every correction from 0 through $L$ must still be present; otherwise the sum does not estimate $\mathbb{E}[Q_L]$.


## 3. Synthetic model used in this notebook

We use the same manufactured model developed in `synthetic_linear_model.ipynb`. The model is simple enough to analyze exactly, but every sample still requires constructing and solving sparse linear systems.

### Model hierarchy

The model is driven by one scalar standard-normal random variable

$$
X\sim N(0,1).
$$

The limiting random quantity we want to approximate is

$$
Q(X)=X.
$$

At level $\ell$, the model contains

$$
n_\ell=4*2^\ell
$$

unknowns and has step size

$$
h_\ell=\frac{1}{n_\ell}.
$$

For the first four levels,

$$
n_0=4,\qquad n_1=8,\qquad n_2=16,\qquad n_3=32,
$$

and

$$
h_0=\frac14,\qquad
h_1=\frac18,\qquad
h_2=\frac1{16},\qquad
h_3=\frac1{32}.
$$

The number of unknowns doubles at each level, while the step size is halved. Therefore,

$$
h_\ell\to 0
$$

as the level becomes finer.

### Finite-level approximation

The scalar approximation at level $\ell$ is

$$
q_\ell(X)
=
X+h_\ell\sqrt{10^{-4}+|X|}.
$$

This can be separated into the limiting quantity and a level-dependent approximation error:

$$
q_\ell(X)
=
\underbrace{X}_{\text{limiting quantity}}
+
\underbrace{h_\ell\sqrt{10^{-4}+|X|}}_{\text{level error}}.
$$

Because $h_\ell\to 0$,

$$
q_\ell(X)\to X=Q(X).
$$

Thus, finer levels provide increasingly accurate approximations of the limiting random quantity.

### Manufactured linear system

We choose the exact level-$\ell$ solution to be the uniform vector

$$
u_\ell^*(X)
=
q_\ell(X)\mathbf{1}_{n_\ell},
$$

where $\mathbf{1}_{n_\ell}$ is a vector containing $n_\ell$ ones.

The level matrix is

$$
A_\ell=(\ell+2)I_{n_\ell}.
$$

This matrix is sparse, symmetric, and positive definite. The right-hand side is constructed from the known solution:

$$
\begin{aligned}
b_\ell(X)
&=A_\ell u_\ell^*(X)\\
&=(\ell+2)q_\ell(X)\mathbf{1}_{n_\ell}.
\end{aligned}
$$

The complete system is therefore

$$
(\ell+2)I_{n_\ell}u_\ell(X)
=
(\ell+2)q_\ell(X)\mathbf{1}_{n_\ell}.
$$

Solving it recovers

$$
u_\ell(X)
=
q_\ell(X)\mathbf{1}_{n_\ell}
=
u_\ell^*(X)
$$

up to floating-point error.

The matrix depends only on the level, so it is constructed once and reused. The right-hand side depends on $X$, so it is constructed separately for every sample.

### Quantity of interest

The quantity of interest is the mean of the solution vector:

$$
Q_\ell(X)
=
\frac{1}{n_\ell}
\mathbf{1}_{n_\ell}^{\mathsf T}u_\ell(X).
$$

Because every entry of $u_\ell(X)$ equals $q_\ell(X)$,

$$
Q_\ell(X)=q_\ell(X).
$$

Therefore,

$$
Q_\ell(X)
=
X+h_\ell\sqrt{10^{-4}+|X|}.
$$

The runner ultimately estimates the expectation of this quantity at the selected finest level.

### Coupled MLMC corrections

For a positive correction level $\ell$, the fine and coarse evaluations use the same sampled value $X$:

$$
Y_\ell
=
Q_\ell(X)-Q_{\ell-1}(X).
$$

Substituting the level approximations gives

$$
\begin{aligned}
Y_\ell
&=
\left[
X+h_\ell\sqrt{10^{-4}+|X|}
\right]
-
\left[
X+h_{\ell-1}\sqrt{10^{-4}+|X|}
\right]\\
&=
(h_\ell-h_{\ell-1})
\sqrt{10^{-4}+|X|}.
\end{aligned}
$$

The shared random term $X$ cancels because the fine and coarse inputs come from the same realization.

Since

$$
h_{\ell-1}=2h_\ell,
$$

we can also write

$$
Y_\ell
=
-h_\ell\sqrt{10^{-4}+|X|},
\qquad \ell>0.
$$

The positive-level corrections are therefore negative, and their magnitudes decrease as the level becomes finer. This strong coupling also gives the fine-level corrections much smaller variance than the level-zero quantity.

At level zero, there is no coarse model:

$$
Y_0=Q_0.
$$

Only one system is solved for a level-zero sample. At every positive correction level, the runner solves both the fine system at level $\ell$ and the coupled coarse system at level $\ell-1$.

This gives us an inexpensive model with known solutions, genuine fine and coarse linear solves, and correction behavior that is easy to verify.

In [1]:
from dataclasses import dataclass

import numpy as np
from scipy import sparse

from mlmc_linear_systems.linear_solver import (
    LinearSolveResult,
    LinearSystem,
    solve_linear_system,
)
from mlmc_linear_systems.mlmc_model import (
    CoupledInputs,
    MultilevelModel,
)
from mlmc_linear_systems.mlmc_runner import MLMCRunner


### 3.1 Recreate the user model

The runner accepts any object satisfying `MultilevelModel[RandomnessT, ModelInputT]`. It does not require this particular class or inherit from a package base class.

For this example:

| Protocol type | Concrete type | Meaning |
|---|---|---|
| `RandomnessT` | `float` | One sampled value $X$ |
| `ModelInputT` | `SyntheticModelInput` | Level-ready input containing $X$ |

The deterministic matrices are constructed once when the model is created. The sample-dependent right-hand sides are constructed later, whenever the runner requests a correction.


In [2]:
@dataclass(frozen=True)
class SyntheticModelInput:
    """Random input used to construct one level system."""

    random_value: float


@dataclass(frozen=True)
class SyntheticLevel:
    """Deterministic data stored for one model level."""

    index: int
    size: int
    step: float
    matrix: sparse.csr_matrix


class SyntheticLinearModel:
    """Manufactured hierarchy with known level solutions."""

    def __init__(self, number_of_levels: int):
        if number_of_levels <= 0:
            raise ValueError("number_of_levels must be positive.")

        self.number_of_levels = number_of_levels
        self.levels = tuple(
            self._create_level(level)
            for level in range(number_of_levels)
        )

    @staticmethod
    def _create_level(level: int) -> SyntheticLevel:
        size = 4 * 2**level
        step = 1.0 / size
        matrix = sparse.eye(
            size,
            format="csr",
            dtype=float,
        ) * float(level + 2)

        return SyntheticLevel(
            index=level,
            size=size,
            step=step,
            matrix=matrix,
        )

    def _check_level(self, level: int) -> None:
        if level < 0 or level >= self.number_of_levels:
            raise ValueError(
                f"level must be between 0 and "
                f"{self.number_of_levels - 1}."
            )

    def _get_level(self, level: int) -> SyntheticLevel:
        self._check_level(level)
        return self.levels[level]

    def sample_randomness(
        self,
        fine_level: int,
        rng: np.random.Generator,
    ) -> float:
        """Draw the shared standard-normal value for one correction."""
        self._check_level(fine_level)
        return float(rng.normal())

    def couple_inputs(
        self,
        fine_level: int,
        randomness: float,
    ) -> CoupledInputs[SyntheticModelInput]:
        """Construct adjacent inputs from the same random value."""
        self._check_level(fine_level)
        fine_input = SyntheticModelInput(random_value=randomness)

        if fine_level == 0:
            return CoupledInputs(fine=fine_input, coarse=None)

        coarse_input = SyntheticModelInput(random_value=randomness)
        return CoupledInputs(
            fine=fine_input,
            coarse=coarse_input,
        )

    def level_value(
        self,
        level: int,
        model_input: SyntheticModelInput,
    ) -> float:
        level_data = self._get_level(level)
        random_value = model_input.random_value
        level_error = level_data.step * np.sqrt(
            1e-4 + abs(random_value)
        )
        return float(random_value + level_error)

    def build_linear_system(
        self,
        level: int,
        model_input: SyntheticModelInput,
    ) -> LinearSystem:
        """Construct the sample-dependent system at one level."""
        level_data = self._get_level(level)
        level_value = self.level_value(level, model_input)
        expected_solution = np.full(
            level_data.size,
            level_value,
        )
        right_hand_side = np.asarray(
            level_data.matrix @ expected_solution
        )
        return LinearSystem(
            A=level_data.matrix,
            b=right_hand_side,
        )

    def quantity_of_interest(
        self,
        level: int,
        solution: np.ndarray,
        model_input: SyntheticModelInput,
    ) -> float:
        """Return the mean solution as the scalar output."""
        level_data = self._get_level(level)
        solution_vector = np.asarray(solution)
        expected_shape = (level_data.size,)

        if solution_vector.shape != expected_shape:
            raise ValueError(
                f"Solution for level {level} must have shape "
                f"{expected_shape}."
            )
        return float(np.mean(solution_vector))


## 4. Construct the model, solver, and runner

We create four model levels, indexed 0 through 3. The annotation using 
```python
mlmc_model: MultilevelModel[
    float,
    SyntheticModelInput,
] = model
```
is optional at runtime; it documents that the concrete model supplies the interface expected by the runner.

The runner expects a solver callable with the transformation

```text
LinearSystem -> LinearSolveResult
```

We explicitly wrap `solve_linear_system()` to show where solver configuration belongs. This example selects the direct solver. A CG configuration could instead be placed inside the same wrapper without modifying the model or runner.

Finally, `base_seed` identifies this reproducible MLMC experiment. The runner does not use one shared mutable generator. It combines the base seed with each correction level and sample index to reconstruct that task's generator.


In [3]:
number_of_levels = 4
base_seed = 42

model = SyntheticLinearModel(
    number_of_levels=number_of_levels
)

# creates a type-annotated reference to the existing model 
# to tell a static type checker to treat the mlmc-model as 
# an object satisfying the MultilevelModel protocol
mlmc_model: MultilevelModel[
    float,
    SyntheticModelInput,
] = model


# This will be replaced later when the solver interface is updated
def direct_system_solver(
    system: LinearSystem,
) -> LinearSolveResult:
    """Solve one runner-supplied system using the direct method."""
    return solve_linear_system(
        system,
        method="direct",
    )


runner = MLMCRunner(
    mlmc_model,
    solver=direct_system_solver,
    base_seed=base_seed,
)


## 5. Choose fixed sample counts

For the first run, the user chooses the counts directly:

$$
(N_0,N_1,N_2,N_3)=(5 \times 10^4,\, 4000,\, 1000,\,200).
$$

The entry at position $\ell$ is the number of independent samples of $Y_\ell$. Thus, this request schedules 50,000 evaluations of $Y_0=Q_0$, but only 200 evaluations of $Y_3=Q_3-Q_2$.

The initial runner requires at least two samples at every selected level so each unbiased sample variance is defined. These counts are for MLMC samples, not iterations taken by the linear solver.

Because we will leave `finest_level=None`, the runner uses the model's finest available level:

$$
L=\texttt{number\_of\_levels}-1=3.
$$

Consequently, the count sequence must have exactly four entries and the returned estimator targets $\mathbb{E}[Q_3]$.


In [4]:
samples_per_level = [50000, 4000, 1000, 200]

for level, sample_count in enumerate(samples_per_level):
    print(
        f"level {level}: {sample_count} samples of Y_{level}"
    )


level 0: 50000 samples of Y_0
level 1: 4000 samples of Y_1
level 2: 1000 samples of Y_2
level 3: 200 samples of Y_3


## 6. Run the estimator

`run_fixed()` performs the complete serial execution. For each pair `(level, sample_index)`, it:

1. constructs the deterministic RNG assigned to that task;
2. asks `compute_sample_correction()` for one coupled correction;
3. solves one system for level 0 or two adjacent systems for a positive level;
4. updates the matching online correction and elapsed-time statistics; and
5. discards the returned solution arrays after their scalar data have been consumed.

The method returns one immutable `MLMCResult`. The runner internally retains mutable `LevelStatistics` so additional samples can be added later, but the returned result is a snapshot and will not change.


In [5]:
initial_result = runner.run_fixed(samples_per_level)

print("selected finest level:", initial_result.finest_level)
print("number of level results:", len(initial_result.level_results))


selected finest level: 3
number of level results: 4


## 7. Interpret the run-level result

The object returned by `run_fixed()` is an `MLMCResult`. It represents one completed snapshot of the entire multilevel run, rather than the result of a single correction level.

### 7.1 Structure of `MLMCResult`

`MLMCResult` has two stored fields:

| Field | Meaning |
|---|---|
| `finest_level` | The finest correction level $L$ included in this run and therefore the finite-level target $Q_L$ |
| `level_results` | An ordered tuple of `LevelResult` snapshots for correction levels $0,1,\ldots,L$ |

For a run with `finest_level=3`, `level_results` contains four entries. The entry at position $\ell$ contains the statistics for $Y_\ell$. Keeping the entries in level order lets the run-level result preserve both the complete correction breakdown and the hierarchy used to form the estimator.

The `MLMCResult` object returned after a run is an immutable snapshot. The runner may later update its internal `LevelStatistics` through `add_samples()`, but it returns a new `MLMCResult`; the previously returned result does not change.

The class also exposes four derived properties:

| Property | Run-level meaning |
|---|---|
| `estimate` | Sum of the correction-level sample means |
| `estimator_variance` | Sum of the estimated variances of those sample means |
| `standard_error` | Square root of the estimator variance |
| `total_cost` | Sum of the measured successful correction-evaluation times over all levels |

These aggregate quantities are calculated from `level_results` whenever the properties are accessed. 

#### MLMC estimate

The `estimate` property adds the mean correction from every `LevelResult`,

$$
\widehat Q_{\mathrm{MLMC}}
=\sum_{\ell=0}^{L}\overline{Y}_\ell.
$$

This is the run's estimate of $\mathbb{E}[Q_L]$. The `finest_level` field records which finite-level quantity the sum targets.

#### Estimator variance

Each `LevelResult` contains `variance_of_mean`, which is $s_\ell^2/N_\ell$. Because independently sampled correction means are combined, the `estimator_variance` property adds these values:

$$
\widehat{\operatorname{Var}}
\left[\widehat Q_{\mathrm{MLMC}}\right]
=\sum_{\ell=0}^{L}\frac{s_\ell^2}{N_\ell}.
$$

#### Standard error

The `standard_error` property is the square root of the estimated variance of the complete MLMC mean:

$$
\widehat{\operatorname{SE}}
=\sqrt{\widehat{\operatorname{Var}}[\widehat Q_{\mathrm{MLMC}}]}.
$$

This measures estimated sampling uncertainty. It does not include the discretization bias between $Q_L$ and the limiting quantity $Q$.

#### Total measured cost

The `total_cost` property adds `total_sample_cost` from every `LevelResult`. It is the total measured elapsed time spent obtaining corrections across the complete hierarchy. Timing depends on the current machine and workload, so it is not expected to be reproducible even when the correction values are reproducible.


In [6]:
print(
    f"estimate of E[Q_{initial_result.finest_level}]: "
    f"{initial_result.estimate:.8f}"
)
print(
    "estimated variance of the MLMC mean: "
    f"{initial_result.estimator_variance:.8e}"
)
print(
    "estimated standard error: "
    f"{initial_result.standard_error:.8e}"
)
print(
    "summed successful correction evaluation time: "
    f"{initial_result.total_cost:.6f} seconds"
)


estimate of E[Q_3]: 0.02756403
estimated variance of the MLMC mean: 2.14200733e-05
estimated standard error: 4.62818251e-03
summed successful correction evaluation time: 2.715502 seconds


> **Timing note:** `total_cost` is the sum of the measured evaluation times
> of all successful correction samples. Each correction time includes model
> randomness, input coupling, system construction, fine and optional coarse
> solves, and quantity-of-interest evaluation. It does not include runner
> validation, deterministic RNG construction, statistics updates, result
> construction, or other orchestration overhead. It is therefore a measure
> of total correction work, not the complete wall-clock duration of the
> runner. Under future parallel execution, summed correction cost may be
> larger than the actual elapsed wall time because correction tasks can
> overlap.

### 7.2 Estimator Evaluation

The limiting quantity of interest is

$$
Q(X)=X,
$$

where

$$
X\sim N(0,1).
$$

Because a standard-normal random variable has mean zero,

$$
\mathbb{E}[Q]
=
\mathbb{E}[X]
=
0.
$$

However, the current runner does not evaluate the limiting quantity directly. With finest level $L$, it estimates the finite-level quantity

$$
Q_L(X)
=
X+h_L\sqrt{10^{-4}+|X|}.
$$

Taking the expectation gives

$$
\begin{aligned}
\mathbb{E}[Q_L]
&=
\mathbb{E}[X]
+
h_L\mathbb{E}\left[
\sqrt{10^{-4}+|X|}
\right]
\\
&=
h_L\mathbb{E}\left[
\sqrt{10^{-4}+|X|}
\right],
\end{aligned}
$$

because $\mathbb{E}[X]=0$.

To calculate this finite-level reference value, recall that the expectation of a function $g(X)$ of a continuous random variable is

$$
\mathbb{E}[g(X)]
=
\int_{-\infty}^{\infty}
g(x)f_X(x)\,dx,
$$

where $f_X(x)$ is the probability density of $X$.

For a standard-normal random variable,

$$
f_X(x)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{x^2}{2}\right).
$$

Therefore,

$$
\mathbb{E}\left[
\sqrt{10^{-4}+|X|}
\right]
=
\int_{-\infty}^{\infty}
\sqrt{10^{-4}+|x|}
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{x^2}{2}\right)
\,dx.
$$

Both $\sqrt{10^{-4}+|x|}$ and the standard-normal density are symmetric about zero. We can therefore write the integral as

$$
\mathbb{E}\left[
\sqrt{10^{-4}+|X|}
\right]
=
2
\int_0^\infty
\sqrt{10^{-4}+x}
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{x^2}{2}\right)
\,dx.
$$

This integral does not have a simple elementary expression, so we evaluate it numerically. Its value is approximately

$$
\mathbb{E}\left[
\sqrt{10^{-4}+|X|}
\right]
\approx 0.82226443.
$$

For this run, the finest level is $L=3$, where

$$
h_3=\frac{1}{32}.
$$

The finite-level expectation is therefore

$$
\mathbb{E}[Q_3]
=
\frac{1}{32}(0.82226443)
\approx 0.02569576.
$$

The MLMC estimate should approach $0.02569576$ as the sample counts increase.

There are two different errors to distinguish:

- The difference between the MLMC estimate and $\mathbb{E}[Q_3]$ is sampling error.
- The difference between $\mathbb{E}[Q_3]$ and the limiting value $\mathbb{E}[Q]=0$ is discretization bias.

The numerical integration below is used only to calculate a reference value for this teaching example. It is not performed by the MLMC runner.

In [7]:
from scipy.integrate import quad


def expected_error_integrand(x: float) -> float:
    """Return the positive-half-line normal expectation integrand."""
    normal_density = (
        np.exp(-0.5 * x**2)
        / np.sqrt(2.0 * np.pi)
    )

    return (
        2.0
        * np.sqrt(1e-4 + x)
        * normal_density
    )


expected_error_factor, quadrature_error = quad(
    expected_error_integrand,
    0.0,
    np.inf,
)

finest_level = initial_result.finest_level
finest_step = model.levels[finest_level].step

finite_level_reference = (
    finest_step * expected_error_factor
)

limiting_expectation = 0.0

sampling_difference = (
    initial_result.estimate
    - finite_level_reference
)

discretization_bias = (
    finite_level_reference
    - limiting_expectation
)

In [8]:
print(
    f"MLMC estimate of E[Q_{finest_level}]: "
    f"{initial_result.estimate:.8f}"
)
print(
    f"reference E[Q_{finest_level}]: "
    f"{finite_level_reference:.8f}"
)
print(
    f"limiting E[Q]: "
    f"{limiting_expectation:.8f}"
)
print(
    f"estimate minus finite-level reference: "
    f"{sampling_difference:.8f}"
)
print(
    f"finite-level discretization bias: "
    f"{discretization_bias:.8f}"
)
print(
    "estimated variance of the MLMC mean: "
    f"{initial_result.estimator_variance:.8e}"
)
print(
    "estimated standard error: "
    f"{initial_result.standard_error:.8e}"
)

MLMC estimate of E[Q_3]: 0.02756403
reference E[Q_3]: 0.02569576
limiting E[Q]: 0.00000000
estimate minus finite-level reference: 0.00186826
finite-level discretization bias: 0.02569576
estimated variance of the MLMC mean: 2.14200733e-05
estimated standard error: 4.62818251e-03


### 7.3 Inspect every correction level

Each LevelResult represents either $Y_0=Q_0$ or $Y_\ell=Q_\ell-Q_{\ell-1}$ for $\ell>0$.. It contains no fine or coarse solution arrays. Its fields are:

| Field | Meaning |
|---|---|
| `level` | Correction index $\ell$ |
| `sample_count` | Number $N_\ell$ of completed corrections |
| `mean_correction` | Sample mean $\overline{Y}_\ell$ |
| `sample_variance` | Unbiased correction variance $s_\ell^2$ |
| `variance_of_mean` | Estimated variance $s_\ell^2/N_\ell$ |
| `mean_sample_cost` | Mean elapsed time for one correction |
| `total_sample_cost` | Total elapsed time at the level |

For this manufactured model, positive-level corrections are negative because $h_\ell<h_{\ell-1}$. Their magnitudes and variances should decrease as the levels become finer. Level 0 behaves differently: $Y_0=Q_0$ still contains the full standard-normal term $X$ and therefore has much larger variance.


In [9]:
print(
    f"{'level':>5} {'count':>7} {'mean Y_l':>14} "
    f"{'var(Y_l)':>14} {'var(mean)':>14} {'mean cost':>14}"
)

for level_result in initial_result.level_results:
    print(
        f"{level_result.level:5d} "
        f"{level_result.sample_count:7d} "
        f"{level_result.mean_correction:14.6e} "
        f"{level_result.sample_variance:14.6e} "
        f"{level_result.variance_of_mean:14.6e} "
        f"{level_result.mean_sample_cost:14.6e}"
    )


level   count       mean Y_l       var(Y_l)      var(mean)      mean cost
    0   50000   2.051822e-01   9.980337e-01   1.996067e-05   4.469733e-05
    1    4000  -1.024214e-01   1.947379e-03   4.868447e-07   9.455982e-05
    2    1000  -5.026916e-02   4.468954e-04   4.468954e-07   8.513713e-05
    3     200  -2.492760e-02   1.051319e-04   5.256595e-07   8.629650e-05


### 7.4 Verify how the aggregate properties are formed

The combined values are calculated from the per-level snapshots rather than stored twice. This avoids two copies of the same information becoming inconsistent. We can reconstruct each aggregate directly and verify the result properties.


In [10]:
sum_of_means = sum(
    level_result.mean_correction
    for level_result in initial_result.level_results
)
sum_of_mean_variances = sum(
    level_result.variance_of_mean
    for level_result in initial_result.level_results
)
sum_of_costs = sum(
    level_result.total_sample_cost
    for level_result in initial_result.level_results
)

assert np.isclose(initial_result.estimate, sum_of_means)
assert np.isclose(
    initial_result.estimator_variance,
    sum_of_mean_variances,
)
assert np.isclose(
    initial_result.standard_error,
    np.sqrt(sum_of_mean_variances),
)
assert np.isclose(initial_result.total_cost, sum_of_costs)

print("All run-level aggregates match the level snapshots.")


All run-level aggregates match the level snapshots.


## 8. Deliberately select a lower finest level

A model may provide more levels than a particular experiment needs. Supplying

```python
finest_level=2
```

selects the complete correction hierarchy $Y_0,Y_1,Y_2$ and therefore targets $\mathbb{E}[Q_2]$. The initial count sequence must then contain exactly three positive entries.

This is different from silently omitting a correction. For example, estimating $Q_3$ requires all four terms

$$
\mathbb{E}[Q_3]
=\mathbb{E}[Y_0]+\mathbb{E}[Y_1]+\mathbb{E}[Y_2]+\mathbb{E}[Y_3].
$$

A new runner is used because `run_fixed()` starts a new experiment and never replaces an active run. We reuse the base seed so tasks with the same `(level, sample_index)` identities receive the same random streams.


In [11]:
lower_runner = MLMCRunner(
    mlmc_model,
    solver=direct_system_solver,
    base_seed=base_seed,
)

lower_result = lower_runner.run_fixed(
    samples_per_level[0:3],
    finest_level=2,
)

assert lower_result.finest_level == 2

for lower_level, full_level in zip(
    lower_result.level_results,
    initial_result.level_results[:3],
    strict=True,
):
    assert lower_level.sample_count == full_level.sample_count
    assert np.isclose(
        lower_level.mean_correction,
        full_level.mean_correction,
    )

In [12]:
print(
    "Verified the lower-finest-level run:\n"
    "- The run targeted Q_2 using correction levels 0, 1, and 2.\n"
    "- Each shared level used the same sample count as the full run.\n"
    "- With the same base seed, all shared correction means matched.\n"
    "- Omitting level 3 did not change the random samples assigned "
    "to levels 0 through 2.\n"
)

Verified the lower-finest-level run:
- The run targeted Q_2 using correction levels 0, 1, and 2.
- Each shared level used the same sample count as the full run.
- With the same base seed, all shared correction means matched.
- Omitting level 3 did not change the random samples assigned to levels 0 through 2.



In [13]:
finest_correction_mean = (
    initial_result.level_results[3].mean_correction
)

assert np.isclose(
    initial_result.estimate,
    lower_result.estimate + finest_correction_mean,
)

print(
    f"Estimate targeting E[Q_2]: {lower_result.estimate:.8f}\n"
    f"Mean level-3 correction:    {finest_correction_mean:.8f}\n"
    f"Estimate targeting E[Q_3]: {initial_result.estimate:.8f}\n\n"
    "Adding the level-3 correction changes the Q_2 estimate "
    "into the Q_3 estimate."
)

Estimate targeting E[Q_2]: 0.05249163
Mean level-3 correction:    -0.02492760
Estimate targeting E[Q_3]: 0.02756403

Adding the level-3 correction changes the Q_2 estimate into the Q_3 estimate.


## 9. Reproduce the run

Every correction sample is assigned a fixed identity:

```text
base_seed + correction level + sample index
                         |
                         v
             deterministic Generator
```

The runner passes that generator to `model.sample_randomness()`. The model decides which distribution to sample—in this example it calls `rng.normal()`—while the runner determines which reproducible stream belongs to the task.

Creating a second runner with the same configuration and request should reproduce the same correction statistics


In [14]:
repeated_runner = MLMCRunner(
    mlmc_model,
    solver=direct_system_solver,
    base_seed=base_seed,
)
repeated_result = repeated_runner.run_fixed(samples_per_level)

for first, repeated in zip(
    initial_result.level_results,
    repeated_result.level_results,
    strict=True,
):
    assert first.sample_count == repeated.sample_count
    assert first.mean_correction == repeated.mean_correction
    assert first.sample_variance == repeated.sample_variance

assert initial_result.estimate == repeated_result.estimate
print("The correction statistics were reproduced exactly.")


The correction statistics were reproduced exactly.


## 10. Add samples to the active run

`add_samples()` extends the hierarchy already owned by the runner. It accepts one nonnegative count for every active correction level. A zero means that level receives no new work in this call.

We request

$$
(\Delta N_0,\Delta N_1,\Delta N_2,\Delta N_3)
=(0,100,50,10).
$$

Level 0 remains at 50,000 samples. The other levels continue from their current sample counts. For example, level 3 initially used sample indices 0 through 199, so its ten new samples use indices 200 through 209. No previous random stream is reused.

At least one additional count must be positive. An all-zero request is rejected because it schedules no work.


In [15]:
additional_samples = [0, 100, 50, 10]
updated_samples = [init_sample + added_sample for init_sample, added_sample in zip(samples_per_level, additional_samples)]

updated_result = runner.add_samples(additional_samples)

print(f"{'level':>5} {'initial':>10} {'updated':>10}")
for initial_level, updated_level in zip(
    initial_result.level_results,
    updated_result.level_results,
    strict=True,
):
    print(
        f"{initial_level.level:5d} "
        f"{initial_level.sample_count:10d} "
        f"{updated_level.sample_count:10d}"
    )

assert [
    item.sample_count for item in updated_result.level_results
] == updated_samples


level    initial    updated
    0      50000      50000
    1       4000       4100
    2       1000       1050
    3        200        210


### 10.1 Immutable snapshots and one-shot equivalence

`initial_result` is an immutable snapshot. Adding samples changed the runner's internal accumulators and returned `updated_result`, but it did not alter the earlier result. This is important when intermediate estimates are stored for convergence studies or reports.

The deterministic task identity also guarantees that running $N_\ell$ samples and later adding $M_\ell$ samples produces the same correction sequence as requesting $N_\ell+M_\ell$ samples at once. We verify that property using a new runner. Again, measured costs are not compared.


In [16]:
assert [
    item.sample_count for item in initial_result.level_results
] == samples_per_level

one_shot_runner = MLMCRunner(
    mlmc_model,
    solver=direct_system_solver,
    base_seed=base_seed,
)
one_shot_result = one_shot_runner.run_fixed(
    updated_samples
)

for continued, one_shot in zip(
    updated_result.level_results,
    one_shot_result.level_results,
    strict=True,
):
    assert continued.sample_count == one_shot.sample_count
    assert continued.mean_correction == one_shot.mean_correction
    assert continued.sample_variance == one_shot.sample_variance

print("The continued and one-shot correction statistics match.")


The continued and one-shot correction statistics match.


## 11. Common fixed-run mistakes

### Omitting counts without selecting a lower finest level

For a four-level model, this is invalid:

```python
new_runner.run_fixed([2000, 400, 100])
```

With `finest_level=None`, the target is the model's finest level 3, so four counts are required. To target level 2 deliberately, write:

```python
new_runner.run_fixed(
    [2000, 400, 100],
    finest_level=2,
)
```

### Using zero in the initial request

Every selected correction requires at least two initial samples. Zero is allowed only when adding samples to a level whose statistics already exist.

### Calling `run_fixed()` twice

A second call is rejected rather than interpreted as either a reset or an additional batch. Use `add_samples()` to continue the active run, or construct a new `MLMCRunner` for an independent run.

### Expecting identical timings

The base seed reproduces correction values, not wall-clock timings. Compare estimates, correction means, and variances when checking reproducibility.

### Treating the current runner as adaptive

The current implementation executes fixed user-supplied counts. `run_to_tolerance()` is intentionally deferred until variance-and-cost-based allocation and bias handling are designed and tested.


## 12. What this example verified

This notebook used the complete serial fixed-sample workflow and verified that:

- the user model owns randomness sampling, coupling, system construction, and the quantity of interest;
- the runner owns the base seed, sample identities, solver selection, scheduling, and accumulation;
- a default run includes every model level and targets the model's finest approximation;
- an explicit `finest_level` selects a complete lower telescoping hierarchy;
- `MLMCResult` provides both per-level snapshots and combined estimator properties;
- repeating the same experiment reproduces correction statistics exactly;
- `add_samples()` permits zeros, continues sample indices, and preserves earlier result snapshots; and
- incremental sampling agrees with the equivalent one-shot request.

The next runner milestone is not a different correction calculation. It is an adaptive execution policy that uses the same deterministic task identities and online statistics to choose additional sample counts from estimated correction variances and costs.
